# Week 33, Day 5: Static vs runtime types

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w033d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from dataclasses import dataclass

@dataclass
class Member:
    name: str
    age: int

m = Member(42, "forty-two")
print(m, isinstance(m.age, int))
try:
    print(isinstance([1, 2], list[int]))
except TypeError:
    print("TypeError")

In [ ]:
def is_int_list(value: object) -> bool:
    return isinstance(value, list) and all(isinstance(x, int) for x in value)

print(is_int_list([1, 2, 3]), is_int_list([1, "2"]), is_int_list((1, 2)), is_int_list([]))

scores: list[int] = [70, 81]
scores.append("absent")      # nothing stops this when the program runs
print(scores, type(scores))

In [ ]:
import json
from dataclasses import dataclass
from typing import Any

@dataclass
class Config:
    host: str
    port: int
    debug: bool = False

def load_config(text: str) -> Config:
    data: Any = json.loads(text)                      # Any: we don't know yet
    if not isinstance(data, dict):
        raise ValueError("config must be a JSON object")
    host, port, debug = data.get("host"), data.get("port"), data.get("debug", False)
    if not isinstance(host, str) or not host:
        raise ValueError("host must be a non-empty string")
    if not isinstance(port, int) or isinstance(port, bool) or not 1 <= port <= 65535:
        raise ValueError("port must be a whole number from 1 to 65535")
    if not isinstance(debug, bool):
        raise ValueError("debug must be true or false")
    return Config(host, port, debug)                  # from here on, a real Config

print(load_config('{"host": "localhost", "port": 8080}'))
for bad in ['{"host": "x", "port": "8080"}', '[1, 2]', '{"host": "x", "port": 8080, "debug": "yes"}']:
    try:
        load_config(bad)
    except ValueError as err:
        print("ValueError:", err)

In [ ]:
from typing import get_origin, get_args

for hint in [int, list[int], dict[str, float], tuple[int, ...], int | None]:
    print(f"{str(hint):<18} origin={get_origin(hint)}  args={get_args(hint)}")

In [ ]:
import types
from typing import Union, get_args, get_origin

def matches(value: object, hint: object) -> bool:
    origin, args = get_origin(hint), get_args(hint)
    if hint is None or hint is type(None):
        return value is None
    if origin in (Union, types.UnionType):
        return any(matches(value, a) for a in args)
    if origin is list:
        return isinstance(value, list) and all(matches(x, args[0]) for x in value)
    if origin is dict:
        return isinstance(value, dict) and all(matches(k, args[0]) and matches(v, args[1]) for k, v in value.items())
    if hint is float:
        return isinstance(value, (int, float))
    return isinstance(value, hint)

print(matches([1, 2], list[int]), matches([1, "2"], list[int]), matches(None, str | None))
print(matches({"Ada": [70, 81]}, dict[str, list[int]]), matches({"Ada": [70, "x"]}, dict[str, list[int]]), matches(3, float))

In [ ]:
from typing import cast

text = "12"
n = cast(int, text)                 # a promise to the checker, not a conversion
print(repr(n), type(n).__name__)
try:
    print(n + 1)
except TypeError as err:
    print("TypeError:", err)
print(int(text) + 1)                # what was really needed

> **Key idea.** Static checking happens before the program runs and sees every path but not real data; runtime checking happens while it runs and sees real values on the path taken. Generic hints vanish at run time: `isinstance(x, list[int])` is an error, so check the items. Check data where it enters the program (users, files, JSON, the network), convert it into typed objects once, and trust hints inside. `get_origin` and `get_args` take hints apart for checking values against them. `cast` changes nothing at run time; `isinstance` with `raise` (or `assert`) checks and narrows; `TYPE_CHECKING` guards imports needed only for hints.

### Exercise: Lists of what?

Write two checks that work while the program runs:

- `all_of(value: object, kind: type) -> bool`: True if `value` is a list whose items are all instances of `kind` (an empty list counts);
- `clean_ints(values: list[object]) -> list[int]`: the items that are real whole numbers, in order, leaving out everything else, **including** `True` and `False`.

In [ ]:
def all_of(value, kind):
    ___

def clean_ints(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checks():\n    \"checks\"\n    assert all_of([1, 2], int) and not all_of([1, \"2\"], int) and not all_of((1, 2), int) and all_of([], str)\n    assert all_of([\"a\"], str) and not all_of(\"abc\", str) and all_of([True], int)\n    assert clean_ints([1, \"2\", 3.0, True, None, -4, [5]]) == [1, -4] and clean_ints([]) == []")

### Exercise: Values against hints

Write `matches(value, hint)`, which says whether a value fits a hint, supporting:

- plain classes (`int`, `str`, `bool`, your own classes), using `isinstance`; `float` also accepts `int`;
- `None` (or `type(None)`), which only `None` fits;
- unions such as `int | str` and `str | None`;
- `list[X]`, `set[X]` and `dict[K, V]`, checking every item, key and value;
- `tuple[X, ...]` (any length, all `X`) and `tuple[X, Y, Z]` (exactly that many, each its own type).

In [ ]:
import types
from typing import Union, get_args, get_origin

def matches(value, hint):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_matches():\n    \"matching\"\n    assert matches(3, int) and not matches(\"3\", int) and matches(3, float) and matches(2.5, float) and not matches(2.5, int)\n    assert matches(None, None) and matches(None, type(None)) and not matches(0, None)\n    assert matches(\"a\", int | str) and matches(None, str | None) and not matches(3.5, int | str)\n    assert matches([1, 2], list[int]) and not matches([1, \"2\"], list[int]) and not matches((1,), list[int]) and matches([], list[str])\n    assert matches({\"a\", \"b\"}, set[str]) and not matches({1}, set[str])\n    assert matches({\"Ada\": [1, 2]}, dict[str, list[int]]) and not matches({\"Ada\": [1, \"x\"]}, dict[str, list[int]]) and not matches({1: []}, dict[str, list[int]])\n    assert matches((1, 2, 3), tuple[int, ...]) and matches((), tuple[int, ...]) and not matches((1, \"2\"), tuple[int, ...])\n    assert matches((\"Ada\", 31), tuple[str, int]) and not matches((\"Ada\", 31, 1), tuple[str, int]) and not matches([\"Ada\", 31], tuple[str, int])\n    assert matches([None, 3], list[int | None]) and matches({\"x\": (1.5, 2)}, dict[str, tuple[float, ...]])")

### Exercise: A checking wrapper

Write `checked(func)`, which returns a **new function** that checks its positional arguments against `func`'s hints, calls `func`, then checks the result against the return hint. A failure raises `TypeError` with a message like `amount: expected int, got str` or `return: expected str, got int`. Use `matches` from the starter, `typing.get_type_hints`, and `inspect.signature` for the parameter names. Parameters without hints aren't checked.

In [ ]:
import inspect
import types
from typing import Union, get_args, get_origin, get_type_hints

def matches(value, hint):
    origin, args = get_origin(hint), get_args(hint)
    if hint is None or hint is type(None):
        return value is None
    if origin in (Union, types.UnionType):
        return any(matches(value, a) for a in args)
    if origin is list:
        return isinstance(value, list) and all(matches(x, args[0]) for x in value)
    if hint is float:
        return isinstance(value, (int, float))
    return isinstance(value, hint)

def checked(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checked():\n    \"checking\"\n    def pay(name: str, amount: int, note=None) -> str:\n        return f\"{name}: {amount}\"\n    def broken(x: int) -> str:\n        return x\n    def names(items: list[str]) -> int | None:\n        return len(items) or None\n    p = checked(pay)\n    assert p(\"Ada\", 500) == \"Ada: 500\" and p(\"Ada\", 5, 99) == \"Ada: 5\" and checked(names)([]) is None and checked(names)([\"a\"]) == 1\n    for f, args, msg in [(p, (\"Ada\", \"500\"), \"amount: expected int, got str\"), (p, (3, 3), \"name: expected str, got int\"),\n                         (checked(broken), (4,), \"return: expected str, got int\"), (checked(names), ([1],), \"items: expected list[str], got list\")]:\n        try:\n            f(*args)\n            assert False, f\"should raise TypeError: {msg}\"\n        except TypeError as err:\n            assert str(err) == msg, f\"message was {str(err)!r}\"")

### Exercise: Checking a settings file

Write `load_settings(text: str) -> Settings`, where `Settings` is the dataclass in the starter. The text is JSON. Check it at the boundary: it must be an object with a non-empty string `"name"`, a whole number `"volume"` from 0 to 10 (not a `bool`), and optionally a list of strings `"playlists"` (default empty). Any problem raises `ValueError` (including JSON that won't parse, whose error, `json.JSONDecodeError`, is already a kind of `ValueError`). Unknown keys raise `ValueError` too.

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Any

@dataclass
class Settings:
    name: str
    volume: int
    playlists: list[str] = field(default_factory=list)

def load_settings(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"loading\"\n    S = ns[\"Settings\"]\n    assert load_settings('{\"name\": \"Ada\", \"volume\": 7}') == S(\"Ada\", 7, [])\n    assert load_settings('{\"name\": \"Bo\", \"volume\": 0, \"playlists\": [\"Chill\", \"Gym\"]}') == S(\"Bo\", 0, [\"Chill\", \"Gym\"])\n    for bad in ['{\"name\": \"Ada\", \"volume\": \"7\"}', '{\"name\": \"Ada\", \"volume\": true}', '{\"name\": \"\", \"volume\": 3}', '{\"volume\": 3}',\n                '{\"name\": \"Ada\", \"volume\": 11}', '{\"name\": \"Ada\", \"volume\": 3, \"playlists\": [\"a\", 2]}', '[1]', 'not json',\n                '{\"name\": \"Ada\", \"volume\": 3, \"colour\": \"red\"}']:\n        try:\n            load_settings(bad)\n            assert False, f\"{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Bug hunt: A check that can't be made

The validation crashes with `TypeError: isinstance() argument 2 cannot be a parameterized generic`. Fix `valid_tags` so it checks the list and its items, and prints `True False False`.

In [ ]:
def valid_tags(tags: object) -> bool:
    return isinstance(tags, list[str])

print(valid_tags(["news", "sport"]), valid_tags(["news", 3]), valid_tags("news"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"checked\"\n    expect_output(\"True False False\")")

### Bug hunt: A cast that converts nothing

The program crashes with `TypeError: can only concatenate str (not "int") to str`. Someone thought `cast` turns text into a number. Fix the line that uses `cast` so it prints `13`.

In [ ]:
from typing import cast

def next_number(text: str) -> int:
    n = cast(int, text.strip())
    return n + 1

print(next_number(" 12 "))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a real conversion\"\n    expect_output(\"13\")")

### Bug hunt: Trusting the file

The order file stores quantities, but one was saved as text, so the total crashes with `TypeError`. `total_qty` trusts its hint, which is fine inside the program; the bug is that `load_orders` is the boundary and doesn't check. Fix `load_orders` so it converts quantities written as digit strings (like `"3"`) to `int`, raises `ValueError` for anything else that isn't an `int`, and the program prints `6`.

In [ ]:
import json

def load_orders(text: str) -> list[dict[str, int]]:
    return json.loads(text)

def total_qty(orders: list[dict[str, int]]) -> int:
    return sum(o["qty"] for o in orders)

print(total_qty(load_orders('[{"qty": 2}, {"qty": "3"}, {"qty": 1}]')))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the total\"\n    expect_output(\"6\")\n    try:\n        load_orders('[{\"qty\": \"three\"}]')\n        assert False, \"a word should raise ValueError\"\n    except ValueError:\n        pass")

### Puzzle: Checking at the edge

Arrange the lines into a function that checks data loaded from a file before trusting it. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from typing import cast, get_origin, get_args

p = ___   # cast(int, "5")
q = ___   # get_origin(list[str])
r = ___   # get_args(dict[str, int])
s = ___   # get_origin(int)
t = ___   # type([1, 2]) is list

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"5\" and q is list and r == (str, int), \"p to r\"\n    assert s is None and t is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a data checker with paths

Write `validate(value, hint, path="value") -> list[str]`, which checks data (such as JSON already loaded) against a hint and returns **every** problem, each with the path to where it went wrong. An empty list means the data is valid.

- Plain classes are checked with `isinstance`, except that `bool` is **not** accepted for `int` or `float` (in data, it's almost always a mistake), and `int` **is** accepted for `float`. A failure is reported as `value.age: expected int, got str`.
- `X | None` and other unions: valid if any part fits; otherwise `expected int | None, got str` (using `str(hint)`).
- `list[X]`: must be a list (otherwise `expected list[int], got dict`); each item is checked with the path `value[0]`, `value[1]`, ...
- `dict[str, X]`: must be a dict; each value is checked with the path `value['Ada']`.
- A **dataclass** (use `dataclasses.is_dataclass` and `fields`): the value must be a dict (otherwise `value: expected Member, got list`); each field is checked with the path `value.name`; a field without a default that's missing gives `value: missing field 'age'`; keys that aren't fields give `value: unknown field 'colour'` (reported in sorted order, after the fields). Use `typing.get_type_hints` on the class for the fields' hints.

Problems appear in the order the data is walked: fields in the order the class defines them, list items in order, dictionary values in the order of the keys.

In [ ]:
import dataclasses
import types
import typing
from dataclasses import dataclass, field

@dataclass
class Member:
    name: str
    age: int
    email: str | None = None

@dataclass
class Club:
    name: str
    members: list[Member]
    fees: dict[str, float] = field(default_factory=dict)

def validate(value, hint, path="value"):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_simple():\n    \"simple hints\"\n    assert validate(3, int) == [] and validate(True, int) == [\"value: expected int, got bool\"] and validate(2, float) == []\n    assert validate(\"x\", int | None) == [\"value: expected int | None, got str\"] and validate(None, int | None) == []\n    assert validate([1, \"a\", 3, \"b\"], list[int]) == [\"value[1]: expected int, got str\", \"value[3]: expected int, got str\"]\n    assert validate({\"a\": 1}, list[int]) == [\"value: expected list[int], got dict\"]\n    assert validate({\"Ada\": 2.5, \"Bo\": \"x\"}, dict[str, float], \"fees\") == [\"fees['Bo']: expected float, got str\"]\ndef test_dataclasses():\n    \"dataclasses\"\n    M, C = ns[\"Member\"], ns[\"Club\"]\n    assert validate({\"name\": \"Ada\", \"age\": 31}, M) == []\n    assert validate({\"name\": \"Ada\", \"age\": \"31\", \"colour\": \"red\", \"badge\": 1}, M) == [\n        \"value.age: expected int, got str\", \"value: unknown field 'badge'\", \"value: unknown field 'colour'\"]\n    assert validate({\"name\": \"Ada\"}, M, \"m\") == [\"m: missing field 'age'\"] and validate([1], M) == [\"value: expected Member, got list\"]\n    club = {\"name\": \"Chess\", \"members\": [{\"name\": \"Ada\", \"age\": 31}, {\"name\": 7, \"age\": 12, \"email\": 5}, {\"age\": True}],\n            \"fees\": {\"junior\": 5, \"senior\": \"ten\"}}\n    assert validate(club, C, \"club\") == [\n        \"club.members[1].name: expected str, got int\",\n        \"club.members[1].email: expected str | None, got int\",\n        \"club.members[2]: missing field 'name'\",\n        \"club.members[2].age: expected int, got bool\",\n        \"club.fees['senior']: expected float, got str\",\n    ], \"\\n\".join(validate(club, C, \"club\"))")

## Warm-up

A dataclass reads its fields' hints, yet accepts a number for `name` and text for `age` without a murmur. And `isinstance` refuses outright to check `list[int]`. Both show the same thing: hints describe the program for tools that read it **before** it runs, and they mostly disappear **while** it runs. Today is about using each kind of checking for what it's good at.

## Two moments

| | Static checking (before running) | Runtime checking (while running) |
|---|---|---|
| Who does it | a type checker such as mypy, and your editor | your own code: `isinstance`, `if`, `raise` |
| What it sees | the code and its hints, every path through it | the actual values, on the path that runs |
| What it can't see | the values that will really arrive from files, users, the network | paths that don't run, and the contents of hints like `list[int]` |
| Cost | none when the program runs | a little time on every check |
| Finds problems | before anyone runs the code | only when that line runs with bad data |

Neither replaces the other. A type checker proves that, **if** the data is what the hints say, every call fits. It can't prove that a JSON file from somewhere else really contains a number where the hint says `int`. That's runtime checking's job.

## Generics disappear

When a program runs, `list[int]` is just `list`: Python keeps no record of what a list is meant to hold, because a list can hold anything. So `isinstance` can check that something is a list, but not what's in it. To check the contents, check each item:

The same goes for your own generic classes: `Stack[int]()` makes an ordinary `Stack`, and nothing checks what's pushed onto it, except mypy, before the program runs.

## Checking at the boundaries

So where should a program check types while it runs? At its **boundaries**: the places where data comes in from outside, where no type checker can vouch for it.

- text typed by a user, command-line arguments (Week 21);
- files: CSV, JSON, configuration (Week 23);
- data from other programs and from the network.

Check (and convert) the data **once**, at the boundary, into well-typed objects. Inside the program, trust the hints and let the type checker do the rest. That keeps checks in one place instead of scattered `isinstance` calls everywhere.

`json.loads` returns `Any`, honestly: its result depends on the text. Checking turns that `Any` into a `Config` that mypy can then follow everywhere. Libraries such as pydantic do this kind of checking automatically from the hints, which is useful in bigger programs; the idea is the same.

## Checking values against hints

Hints are objects, so a program can read them and check values against them. Two functions from `typing` take hints apart:

- `get_origin(hint)` gives the plain class behind a generic hint (`list` for `list[int]`), or `None` for a plain class like `int`. For a union such as `int | None`, it gives a union type (which one depends on the Python version, so compare with both `typing.Union` and `types.UnionType`).
- `get_args(hint)` gives what's in the brackets.

With these, a short recursive function (Week 15) can check a value against a hint:

Notice `float` accepts an `int`, following the same convention as type checkers. The practice builds this out, and the daily challenge turns it into a checker that reports exactly where data goes wrong.

## Telling the checker what you know

Sometimes you know more than the type checker can work out. There are three tools for telling it, and it's important to know what each does **when the program runs**:

| Tool | To the type checker | When the program runs |
|---|---|---|
| `cast(int, value)` | "treat `value` as an `int` from here" | **nothing**: returns `value` unchanged, unchecked |
| `assert isinstance(value, int)` | narrows `value` to `int` after this line | checks, and raises `AssertionError` if wrong (unless Python runs with `-O`) |
| `if not isinstance(value, int): raise TypeError(...)` | narrows, like the `assert` | checks, always |

`cast` is a promise you make to the checker. If the promise is wrong, nothing catches it, and the program fails somewhere else later. Use it only when you're sure and the checker can't follow (for example, after checking with a helper function it doesn't understand). For data from outside, check with `isinstance` and raise.

One more tool you'll see in real projects: `typing.TYPE_CHECKING` is `False` when the program runs and treated as `True` by type checkers. Imports needed only for hints go inside `if TYPE_CHECKING:`, which avoids slow imports and circular imports between modules (Week 20) at run time. And functions declared to return `TypeGuard[...]` (or `TypeIs[...]` in newer Pythons) tell the checker that a `True` result narrows their argument, as `isinstance` does.

## Practice

## Quiz

## Daily challenge

## Before you go

Find the place in one of your programs where data comes in from outside (a file, user input, JSON) and check whether it's validated there. If it isn't, write the checks, and convert the data into a typed object. Tomorrow is the Forge: a typed program from start to finish.